In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import datetime
import json
import os
import random

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import wandb
from accelerate.commands.config.update import description
from transformers import (
    AutoModelForMaskedLM,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

def set_seed(seed=42) -> None:
    """Set all seeds to make results reproducible (deterministic mode).
    When seed is a false-y value or not supplied, disables deterministic mode."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

from dotenv import load_dotenv
load_dotenv()

True

In [3]:
for i in range(torch.cuda.device_count()):
    print(torch.cuda.get_device_properties(i).name)

# os.environ["MY_VARIABLE"] = "my_value"


NVIDIA RTX A5000
NVIDIA RTX A5000
NVIDIA RTX A5000
NVIDIA RTX A5000


## Load trained model from wandb

In [6]:
def load_model_from_wandb(model_path):
    run = wandb.init()
    artifact = run.use_artifact(model_path, type='model')
    artifact_dir = artifact.download()
    model = AutoModelForMaskedLM.from_pretrained(artifact_dir)
    return model, artifact_dir

def load_model_from_local(model_path):
    model = AutoModelForMaskedLM.from_pretrained(model_path)
    return model, model_path

def load_model(model_path, use_wandb=True):
    if use_wandb:
        return load_model_from_wandb(model_path)
    else:
        return load_model_from_local(model_path)
    
# checking if two hugging face models are same or not
model_from_wandb, model_from_wandb_path = load_model_from_wandb("nasa-impact/mlm-fine-tuning/model-ytxjrbhy:v1")
model_from_local, model_from_local_path = load_model_from_local("/rhome/sawale/indus_traning/mlm-fine-tuning/mlm/tmp/models/timestamp_20241216_18-41-39/nasa-impact/nasa-smd-ibm-v0.1/checkpoint-23000")


# Compare the state_dicts
are_models_identical = all(
    torch.equal(param1, param2) 
    for param1, param2 in zip(model_from_wandb.state_dict().values(), model_from_local.state_dict().values())
)

print(f"Are the models identical? {are_models_identical}")



wandb: Downloading large artifact model-ytxjrbhy:v1, 475.71MB. 4 files... 
wandb:   4 of 4 files downloaded.  
Done. 0:0:0.8


Are the models identical? True
